# Chapter 2: Pewter City -- Randomized Experiments

---

> *"Gym Leader Brock's arena awaits, and Professor Oak has designed a randomized experiment to test whether Pewter Protein supplements improve battle performance. It's time to learn the gold standard of causal inference."*

In this chapter, you will learn:

1. Why **randomization** solves the selection bias problem from Chapter 1
2. How to assess **covariate balance** to verify a good experiment
3. The **difference-in-means** estimator and its properties
4. **Regression adjustment** (Lin, 2013) to improve precision
5. **Randomization inference** (Fisher's exact test) -- a design-based alternative to the t-test
6. **Power analysis** -- how big does your experiment need to be?
7. **Noncompliance** -- what happens when trainers don't follow the assignment?
8. **Attrition** -- what happens when trainers drop out of the study?

---

## 1. Setup & Data Loading

In [ ]:
# Core scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

# Kanto utilities
from kanto_utils import (
    load_protein_rct, apply_kanto_theme, type_color,
    difference_in_means, randomization_inference, balance_table,
    wald_estimator,
    oak_says, blue_says, blues_mistake, badge_earned,
    gym_leader_says, power_analysis_widget,
)

# Apply the Kanto visual theme
apply_kanto_theme()

# Reproducibility
np.random.seed(151)

# Load the Pewter Protein RCT dataset
rct = load_protein_rct()
print(f"Loaded Pewter Protein RCT: {len(rct)} trainers in the trial.")
print(f"Columns: {list(rct.columns)}")
rct.head()

In [ ]:
oak_says(
    "Welcome to Pewter City! Here at the Pewter Gym, I've designed an experiment to test whether "
    "<b>Pewter Protein</b> -- a special supplement -- improves a trainer's performance against "
    "Gym Leader Brock. We <b>randomly assigned</b> 200 trainers to receive the protein (treatment) "
    "or a placebo (control). Randomization is the gold standard because it ensures that, on average, "
    "the treatment and control groups are <b>identical in all respects</b> -- observed and unobserved -- "
    "except for the treatment itself. This eliminates selection bias."
)

In [ ]:
# Quick summary of the experimental design
print("Experimental Design Summary")
print("=" * 50)
print(f"Total trainers enrolled:  {len(rct)}")
print(f"Assigned to treatment:    {rct['treatment_assigned'].sum()}")
print(f"Assigned to control:      {(rct['treatment_assigned'] == 0).sum()}")
print(f"\nOutcome variables:")
print(f"  brock_win:       Binary (1 = defeated Brock, 0 = lost)")
print(f"  damage_to_onix:  Continuous (total HP damage dealt to Brock's Onix)")
print(f"  pokemon_remaining: Count of Pokemon remaining after the battle")

---
## 2. Balance Table: Did Randomization Work?

Before analyzing treatment effects, we should verify that randomization produced **balanced** groups. If the treatment and control groups look similar on pre-treatment covariates, we have evidence that the randomization worked.

We use the **Standardized Mean Difference (SMD)**: $\text{SMD} = \frac{\bar{X}_T - \bar{X}_C}{\sqrt{(S_T^2 + S_C^2)/2}}$

A common rule of thumb: $|\text{SMD}| < 0.1$ indicates good balance.

In [ ]:
# Covariates to check for balance (pre-treatment variables)
covariates = ['team_level', 'trainer_experience', 'strategy_score']

bal = balance_table(
    df=rct,
    treatment_col='treatment_assigned',
    covariate_cols=covariates
)

print("Covariate Balance Table (Treatment vs. Control)")
print("=" * 60)
bal

In [ ]:
# Visualize balance with a dot plot
fig, ax = plt.subplots(figsize=(8, 4))

y_pos = np.arange(len(covariates))
smds = bal['std_diff'].values

colors = ['#4DAD5B' if abs(s) < 0.1 else '#EE1515' for s in smds]
ax.barh(y_pos, np.abs(smds), color=colors, edgecolor='white', height=0.5)
ax.axvline(0.1, color='#FFD733', linestyle='--', linewidth=2, label='|SMD| = 0.1 threshold')
ax.set_yticks(y_pos)
ax.set_yticklabels(covariates)
ax.set_xlabel('|Standardized Mean Difference|')
ax.set_title('Covariate Balance: Treatment vs. Control')
ax.legend(frameon=True)
ax.set_xlim(left=0)

for i, smd in enumerate(smds):
    ax.text(abs(smd) + 0.005, i, f'{abs(smd):.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

all_balanced = all(abs(s) < 0.1 for s in smds)
if all_balanced:
    oak_says(
        "Excellent! All covariates have |SMD| below 0.1. "
        "Randomization appears to have worked well -- the treatment and control groups "
        "are similar on pre-treatment characteristics. Any difference in outcomes "
        "can be credibly attributed to the Pewter Protein treatment."
    )
else:
    oak_says(
        "Some covariates show slight imbalance (|SMD| > 0.1). With only 200 trainers, "
        "some chance imbalance is expected. We can address this with <b>regression adjustment</b> "
        "in Section 4, which controls for baseline covariates to improve precision."
    )

---
## 3. Difference-in-Means Estimator

The simplest and most transparent estimator for a randomized experiment:

$$\hat{\tau} = \bar{Y}_T - \bar{Y}_C = \frac{1}{n_T} \sum_{i: D_i=1} Y_i - \frac{1}{n_C} \sum_{i: D_i=0} Y_i$$

Under randomization, this is an **unbiased** estimator of the ATE.

In [ ]:
# Difference-in-means for both outcomes
print("Difference-in-Means Estimates (Intent-to-Treat)")
print("=" * 60)
print("Note: Using treatment_assigned (ITT) to preserve randomization.\n")

outcomes = {
    'brock_win': 'Defeated Brock (binary)',
    'damage_to_onix': 'Damage to Onix (continuous)',
}

results = {}
for outcome, label in outcomes.items():
    res = difference_in_means(
        y=rct[outcome].values,
        treatment=rct['treatment_assigned'].values
    )
    results[outcome] = res
    print(f"Outcome: {label}")
    print(f"  Estimate (ATE): {res['estimate']:.4f}")
    print(f"  Standard Error: {res['se']:.4f}")
    print(f"  95% CI:         [{res['ci_lower']:.4f}, {res['ci_upper']:.4f}]")
    print(f"  p-value:        {res['p_value']:.4f}")
    sig = "statistically significant" if res['p_value'] < 0.05 else "NOT statistically significant"
    print(f"  Result:         {sig} at alpha = 0.05")
    print()

In [ ]:
# Visualize with grouped bar charts
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (outcome, label) in zip(axes, outcomes.items()):
    treated = rct[rct['treatment_assigned'] == 1][outcome]
    control = rct[rct['treatment_assigned'] == 0][outcome]
    
    means = [control.mean(), treated.mean()]
    sems = [control.sem(), treated.sem()]
    
    bars = ax.bar(
        ['Control\n(Placebo)', 'Treatment\n(Pewter Protein)'],
        means,
        yerr=[1.96 * s for s in sems],
        color=['#3B4CCA', '#EE1515'],
        edgecolor='white',
        width=0.5,
        capsize=8,
        alpha=0.85,
    )
    
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{m:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    res = results[outcome]
    ax.set_title(f'{label}\nATE = {res["estimate"]:.3f} (p = {res["p_value"]:.3f})')
    ax.set_ylabel(label)

fig.suptitle('Pewter Protein RCT: Treatment vs. Control', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
gym_leader_says("Brock",
    "Interesting results. As a Rock-type specialist, I can tell you that the protein supplement "
    "seems to affect how much damage trainers deal to my Onix. But remember -- "
    "the point estimate is just a single number. The confidence interval tells you "
    "the range of plausible effect sizes."
)

---
## 4. Regression Adjustment (Lin, 2013)

Even in a randomized experiment, we can gain **precision** by adjusting for pre-treatment covariates. Lin (2013) showed that the correct way to do this is to include:

1. The treatment indicator $D_i$
2. **Demeaned** covariates $\tilde{X}_i = X_i - \bar{X}$
3. **Interactions** $D_i \times \tilde{X}_i$

$$Y_i = \alpha + \tau D_i + \beta' \tilde{X}_i + \gamma' (D_i \times \tilde{X}_i) + \epsilon_i$$

This gives an unbiased ATE estimate ($\hat{\tau}$) with potentially smaller standard errors.

In [ ]:
# Lin (2013) regression adjustment
covariates_lin = ['team_level', 'trainer_experience', 'strategy_score']

# Demean the covariates
X_demeaned = rct[covariates_lin].copy()
for col in covariates_lin:
    X_demeaned[col] = X_demeaned[col] - X_demeaned[col].mean()

D = rct['treatment_assigned'].values

# Build the Lin regression design matrix
# Treatment + demeaned covariates + interactions
lin_df = pd.DataFrame({'treatment': D})
for col in covariates_lin:
    lin_df[col] = X_demeaned[col].values
    lin_df[f'treatment_x_{col}'] = D * X_demeaned[col].values

lin_X = sm.add_constant(lin_df)

print("Regression Adjustment (Lin, 2013)")
print("=" * 60)
print()

for outcome, label in outcomes.items():
    # Lin regression with HC2 robust standard errors
    model = sm.OLS(rct[outcome].values, lin_X).fit(cov_type='HC2')
    
    tau_lin = model.params['treatment']
    se_lin = model.bse['treatment']
    ci_lin = model.conf_int().loc['treatment']
    p_lin = model.pvalues['treatment']
    
    # Compare to simple difference-in-means
    res_dim = results[outcome]
    
    print(f"Outcome: {label}")
    print(f"  {'Method':<30s} {'Estimate':>10s} {'SE':>10s} {'p-value':>10s}")
    print(f"  {'-'*60}")
    print(f"  {'Difference-in-means':<30s} {res_dim['estimate']:>10.4f} {res_dim['se']:>10.4f} {res_dim['p_value']:>10.4f}")
    print(f"  {'Lin (2013) regression':<30s} {tau_lin:>10.4f} {se_lin:>10.4f} {p_lin:>10.4f}")
    
    precision_gain = (1 - se_lin / res_dim['se']) * 100
    print(f"  Precision gain: {precision_gain:.1f}% reduction in SE")
    print()

In [ ]:
oak_says(
    "The Lin regression gives an estimate very close to the simple difference-in-means "
    "(as it should -- both are unbiased under randomization), but the standard error is often "
    "<b>smaller</b>. This is free precision: by accounting for pre-treatment covariates, we "
    "explain some of the residual variance in outcomes, narrowing our confidence intervals. "
    "The key insight from Lin (2013) is that including interactions $D_i \\times \\tilde{X}_i$ "
    "prevents bias that can arise from simple covariate adjustment."
)

---
## 5. Randomization Inference (Fisher's Exact Test)

The difference-in-means approach relies on large-sample approximations. **Randomization inference** is an alternative that uses the known randomization procedure itself to construct the null distribution.

Under the **sharp null hypothesis** $H_0: Y_i(1) = Y_i(0)$ for all $i$ (treatment has zero effect for every unit), we can permute the treatment labels and recompute the test statistic many times to build a null distribution.

The p-value is the proportion of permutation statistics as extreme as the observed one.

In [ ]:
# Randomization inference on the binary outcome: brock_win
ri_result = randomization_inference(
    y=rct['brock_win'].values,
    treatment=rct['treatment_assigned'].values,
    n_perms=10000,
    seed=151
)

print("Randomization Inference: brock_win")
print("=" * 50)
print(f"Observed difference:  {ri_result['observed_diff']:.4f}")
print(f"Fisher p-value:       {ri_result['p_value']:.4f}")
print(f"Number of permutations: {ri_result['n_perms']:,}")

In [ ]:
# Generate the permutation distribution for visualization
rng = np.random.default_rng(151)
y_bw = rct['brock_win'].values.astype(float)
d_bw = rct['treatment_assigned'].values.astype(float)
n_perms = 10000

null_diffs = np.empty(n_perms)
for i in range(n_perms):
    perm = rng.permutation(d_bw)
    null_diffs[i] = y_bw[perm == 1].mean() - y_bw[perm == 0].mean()

observed_diff = ri_result['observed_diff']

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(null_diffs, bins=50, color='#3B4CCA', alpha=0.6, edgecolor='white',
        density=True, label='Null distribution')
ax.axvline(observed_diff, color='#EE1515', linewidth=2.5, linestyle='-',
           label=f'Observed diff = {observed_diff:.3f}')
ax.axvline(-observed_diff, color='#EE1515', linewidth=2.5, linestyle='--',
           alpha=0.5, label=f'Mirror = {-observed_diff:.3f}')

# Shade the rejection region
ax.fill_betweenx(
    [0, ax.get_ylim()[1] * 1.1],
    observed_diff, null_diffs.max() + 0.05,
    alpha=0.15, color='#EE1515'
)
ax.fill_betweenx(
    [0, ax.get_ylim()[1] * 1.1],
    null_diffs.min() - 0.05, -observed_diff,
    alpha=0.15, color='#EE1515'
)

ax.set_xlabel('Difference in Means (Treatment - Control)')
ax.set_ylabel('Density')
ax.set_title(f'Randomization Inference: brock_win\nFisher p-value = {ri_result["p_value"]:.4f}')
ax.legend(frameon=True, fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
oak_says(
    "Randomization inference is elegant because it makes <b>no distributional assumptions</b>. "
    "We don't need to assume normality or invoke the CLT. The null distribution comes directly "
    "from the experimental design. The p-value tells us: if the treatment truly had zero effect "
    "on every trainer, how often would we see a difference this large just by chance? "
    "Compare this to the t-test p-value from Section 3 -- they should be similar."
)

---
## 6. Power Analysis

Before running an experiment, we need to know: **How many trainers do we need?**

Statistical **power** is the probability of detecting a true effect when one exists:

$$\text{Power} = P\big(\text{reject } H_0 \mid H_1 \text{ is true}\big)$$

Power depends on three things:
- **Sample size** ($n$) -- more trainers = more power
- **Effect size** (Cohen's $d$) -- larger effects are easier to detect
- **Significance level** ($\alpha$) -- stricter thresholds require more data

Use the interactive widget below to explore these trade-offs.

In [ ]:
power_analysis_widget()

In [ ]:
oak_says(
    "The conventional target is 80% power. Notice how rapidly power drops for small effect sizes: "
    "detecting a Cohen's d of 0.1 might require thousands of trainers, while d = 0.5 needs only "
    "about 64 per group. This is why <b>power analysis should be done before data collection</b>. "
    "An underpowered experiment is a waste of everyone's time -- including the Pokemon's."
)

---
## 7. Noncompliance: When Trainers Don't Follow Orders

In an ideal experiment, every trainer assigned to treatment actually takes the protein, and every control trainer abstains. In reality, there is **noncompliance**:

- **Always-takers**: Take the protein regardless of assignment
- **Never-takers**: Refuse the protein regardless of assignment
- **Compliers**: Follow their assignment faithfully

The dataset has a `compliance_type` column and both `treatment_assigned` (the randomized assignment) and `treatment_received` (what actually happened).

In [ ]:
# Explore noncompliance
print("Compliance in the Pewter Protein RCT")
print("=" * 50)
print(f"\nCompliance type distribution:")
print(rct['compliance_type'].value_counts().to_string())

# Cross-tab: assigned vs. received
print("\nTreatment Assignment vs. Treatment Received:")
ct = pd.crosstab(
    rct['treatment_assigned'].map({0: 'Assigned Control', 1: 'Assigned Treatment'}),
    rct['treatment_received'].map({0: 'No Protein', 1: 'Took Protein'}),
    margins=True
)
ct

In [ ]:
blues_mistake(
    claim=(
        "Who needs randomization? I'll just compare trainers who ACTUALLY took the protein "
        "to those who didn't. That's the real treatment effect!"
    ),
    reality=(
        "Comparing by <b>treatment received</b> (per-protocol analysis) reintroduces selection bias! "
        "Trainers who chose to take the protein (always-takers) or refuse it (never-takers) may differ "
        "systematically from compliers. Only the <b>treatment assigned</b> variable is randomized. "
        "Using it gives us the Intent-to-Treat (ITT) estimate, which is causally valid."
    )
)

In [ ]:
# Compare three estimators
print("Three Estimators for the Effect on brock_win")
print("=" * 60)

# 1. Intent-to-Treat (ITT): compare by assignment
itt = difference_in_means(
    y=rct['brock_win'].values,
    treatment=rct['treatment_assigned'].values
)
print(f"\n1. Intent-to-Treat (ITT): compare by ASSIGNMENT")
print(f"   Estimate: {itt['estimate']:.4f}  (SE = {itt['se']:.4f}, p = {itt['p_value']:.4f})")
print(f"   Interpretation: causally valid, but attenuated by noncompliance")

# 2. Per-protocol: compare by what was received (BIASED!)
pp = difference_in_means(
    y=rct['brock_win'].values,
    treatment=rct['treatment_received'].values
)
print(f"\n2. Per-Protocol: compare by RECEIVED treatment")
print(f"   Estimate: {pp['estimate']:.4f}  (SE = {pp['se']:.4f}, p = {pp['p_value']:.4f})")
print(f"   WARNING: potentially biased due to self-selection!")

# 3. Wald (LATE) estimator: ITT / first-stage compliance rate
wald_res = wald_estimator(
    y=rct['brock_win'].values,
    treatment=rct['treatment_received'].values,
    instrument=rct['treatment_assigned'].values
)
print(f"\n3. Wald (LATE) Estimator: ITT / compliance rate")
print(f"   Estimate: {wald_res['estimate']:.4f}  (SE = {wald_res['se']:.4f})")
print(f"   First stage: {wald_res['first_stage']:.4f}")
print(f"   Reduced form: {wald_res['reduced_form']:.4f}")
print(f"   Interpretation: effect on COMPLIERS only (LATE)")

In [ ]:
# Visualize the three estimates
fig, ax = plt.subplots(figsize=(9, 5))

est_names = ['Intent-to-Treat\n(ITT)', 'Per-Protocol\n(BIASED)', 'Wald / LATE\n(Compliers)']
est_values = [itt['estimate'], pp['estimate'], wald_res['estimate']]
est_colors = ['#3B4CCA', '#EE1515', '#4DAD5B']
est_ses = [itt['se'], pp['se'], wald_res['se']]

bars = ax.bar(est_names, est_values, yerr=[1.96 * s for s in est_ses],
              color=est_colors, edgecolor='white', width=0.5, capsize=8, alpha=0.85)

for bar, val in zip(bars, est_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel('Estimated Effect on brock_win')
ax.set_title('Noncompliance: Three Different Estimators')
ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
plt.tight_layout()
plt.show()

oak_says(
    "The ITT is <b>always valid</b> because it uses the randomized assignment. "
    "The per-protocol estimate is biased. The Wald/LATE estimate recovers the effect "
    "for compliers: $\\text{LATE} = \\frac{\\text{ITT}}{\\text{First Stage}} = "
    f"\\frac{{{wald_res['reduced_form']:.3f}}}{{{wald_res['first_stage']:.3f}}} = {wald_res['estimate']:.3f}$. "
    "We'll study instrumental variables in depth in a later chapter."
)

---
## 8. Attrition Analysis

**Attrition** occurs when some participants drop out of the study before the outcome is measured. If attrition is related to treatment assignment, it can bias our results even in a randomized experiment.

Let's simulate attrition in our data and check whether it's differential (related to treatment).

In [ ]:
# Simulate attrition: some trainers don't complete the study
# In the real data, we don't have a completed_study column,
# so we'll create a realistic attrition pattern.
rng_attr = np.random.default_rng(42)

# Attrition is higher for low-level trainers (they give up)
# and slightly higher in the treatment group (supplement side effects)
attrition_prob = (
    0.05  # base rate
    + 0.02 * rct['treatment_assigned']  # slightly higher in treatment
    + 0.03 * (rct['team_level'] < rct['team_level'].quantile(0.25))  # low-level trainers
).values

rct_attr = rct.copy()
rct_attr['completed_study'] = rng_attr.binomial(1, 1 - attrition_prob)

print("Attrition Analysis")
print("=" * 50)
print(f"Total enrolled: {len(rct_attr)}")
print(f"Completed study: {rct_attr['completed_study'].sum()} ({rct_attr['completed_study'].mean():.1%})")
print(f"Attrited: {(rct_attr['completed_study'] == 0).sum()} ({1 - rct_attr['completed_study'].mean():.1%})")

In [ ]:
# Check if attrition is differential: is attrition related to treatment?
attrition_by_group = rct_attr.groupby('treatment_assigned')['completed_study'].agg(['mean', 'sum', 'count'])
attrition_by_group.columns = ['Completion Rate', 'Completed', 'Enrolled']
attrition_by_group.index = ['Control', 'Treatment']
print("Completion rates by assignment group:")
print(attrition_by_group)

# Formal test: is treatment assignment a predictor of attrition?
attrition_test = difference_in_means(
    y=rct_attr['completed_study'].values,
    treatment=rct_attr['treatment_assigned'].values
)
print(f"\nDifference in completion rates:")
print(f"  Estimate: {attrition_test['estimate']:.4f}")
print(f"  p-value:  {attrition_test['p_value']:.4f}")

if attrition_test['p_value'] < 0.05:
    print("\n  WARNING: Statistically significant differential attrition!")
    print("  Analyzing only completers may be biased.")
else:
    print("\n  No evidence of significant differential attrition.")
    print("  Analyzing completers is likely unbiased.")

In [ ]:
# Show how attrition can affect estimates: compare full sample vs completers only
completers = rct_attr[rct_attr['completed_study'] == 1]

full_est = difference_in_means(
    y=rct_attr['brock_win'].values,
    treatment=rct_attr['treatment_assigned'].values
)
comp_est = difference_in_means(
    y=completers['brock_win'].values,
    treatment=completers['treatment_assigned'].values
)

print("Impact of Attrition on Estimates")
print("=" * 50)
print(f"{'Analysis':<25s} {'N':>6s} {'Estimate':>10s} {'SE':>8s}")
print(f"{'-'*49}")
print(f"{'Full sample (all)':<25s} {len(rct_attr):>6d} {full_est['estimate']:>10.4f} {full_est['se']:>8.4f}")
print(f"{'Completers only':<25s} {len(completers):>6d} {comp_est['estimate']:>10.4f} {comp_est['se']:>8.4f}")

oak_says(
    "When attrition is <b>not differential</b> (equal rates in treatment and control), "
    "the completer analysis is generally fine. When attrition IS differential, we worry that "
    "the surviving sample is no longer balanced. Remedies include: (1) Lee (2009) bounds "
    "that bracket the true effect, (2) inverse probability of attrition weighting (IPAW), "
    "or (3) collecting outcome data even for attriters when possible."
)

---
## Challenge Exercises

Put your knowledge to the test with these three challenges.

### Challenge 1: Permutation Test on a Continuous Outcome

Run a randomization inference (permutation) test on `damage_to_onix` (a continuous outcome). Plot the permutation distribution and report the Fisher p-value.

In [ ]:
# CHALLENGE 1: Randomization inference on damage_to_onix
# -------------------------------------------------------

ri_damage = randomization_inference(
    y=rct['damage_to_onix'].values,
    treatment=rct['treatment_assigned'].values,
    n_perms=10000,
    seed=151
)

print("Randomization Inference: damage_to_onix")
print(f"Observed difference:  {ri_damage['observed_diff']:.2f} HP")
print(f"Fisher p-value:       {ri_damage['p_value']:.4f}")

# Generate the permutation distribution for plotting
rng_c1 = np.random.default_rng(151)
y_dmg = rct['damage_to_onix'].values.astype(float)
d_dmg = rct['treatment_assigned'].values.astype(float)

null_dmg = np.empty(10000)
for i in range(10000):
    perm = rng_c1.permutation(d_dmg)
    null_dmg[i] = y_dmg[perm == 1].mean() - y_dmg[perm == 0].mean()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(null_dmg, bins=50, color='#3B4CCA', alpha=0.6, edgecolor='white', density=True,
        label='Null distribution')
ax.axvline(ri_damage['observed_diff'], color='#EE1515', linewidth=2.5,
           label=f'Observed = {ri_damage["observed_diff"]:.2f}')
ax.axvline(-ri_damage['observed_diff'], color='#EE1515', linewidth=2.5,
           linestyle='--', alpha=0.5)
ax.set_xlabel('Difference in Mean Damage to Onix')
ax.set_ylabel('Density')
ax.set_title(f'Randomization Inference: damage_to_onix\np = {ri_damage["p_value"]:.4f}')
ax.legend(frameon=True)
plt.tight_layout()
plt.show()

### Challenge 2: Stratified Analysis by Starter Type

Starter type may modify the treatment effect (trainers with Water-type starters have a type advantage against Brock's Rock-types). Run separate difference-in-means analyses for each starter type and compare the treatment effects.

Does the treatment effect differ by starter type? Which group benefits most?

In [ ]:
# CHALLENGE 2: Stratified analysis by starter_type
# -------------------------------------------------

print("Stratified Analysis: Treatment Effect by Starter Type")
print("=" * 60)
print(f"{'Starter Type':<15s} {'N':>5s} {'ATE':>8s} {'SE':>8s} {'95% CI':>20s} {'p':>8s}")
print("-" * 64)

strat_results = {}
for stype in sorted(rct['starter_type'].unique()):
    subset = rct[rct['starter_type'] == stype]
    res = difference_in_means(
        y=subset['brock_win'].values,
        treatment=subset['treatment_assigned'].values
    )
    strat_results[stype] = res
    ci_str = f"[{res['ci_lower']:.3f}, {res['ci_upper']:.3f}]"
    print(f"{stype:<15s} {len(subset):>5d} {res['estimate']:>8.4f} {res['se']:>8.4f} {ci_str:>20s} {res['p_value']:>8.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
stypes = sorted(strat_results.keys())
x_pos = np.arange(len(stypes))
ates = [strat_results[s]['estimate'] for s in stypes]
ses = [strat_results[s]['se'] for s in stypes]
colors = [type_color(s) for s in stypes]

ax.bar(x_pos, ates, yerr=[1.96 * s for s in ses], color=colors,
       edgecolor='white', width=0.5, capsize=8, alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(stypes)
ax.set_ylabel('Treatment Effect on brock_win')
ax.set_title('Heterogeneous Treatment Effects by Starter Type')
ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')

for i, (ate, se) in enumerate(zip(ates, ses)):
    ax.text(i, ate + 1.96 * se + 0.02, f'{ate:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nInterpretation: Water-type starters already have an advantage against")
print("Brock's Rock types. Check whether the protein supplement provides an")
print("additional boost, or whether it helps Fire-type starters more (catch-up effect).")

### Challenge 3: Simulate an RCT and Plot the Power Curve

Write a simulation that:
1. For each sample size $n \in \{20, 40, 60, \ldots, 500\}$, simulates 1000 RCTs with a true ATE of 0.3
2. In each simulated RCT, tests $H_0: \text{ATE} = 0$ at $\alpha = 0.05$
3. Computes the empirical power (proportion of simulations where $H_0$ is rejected)
4. Plots the power curve and marks the sample size where power first reaches 80%

In [ ]:
# CHALLENGE 3: Simulate power curve
# ----------------------------------

def simulate_power(n_per_group, true_ate=0.3, n_sims=1000, alpha=0.05, seed=151):
    """Simulate n_sims RCTs and return the proportion of significant results."""
    rng = np.random.default_rng(seed)
    rejections = 0
    for _ in range(n_sims):
        # Generate data
        y0 = rng.standard_normal(2 * n_per_group)
        d = np.array([0] * n_per_group + [1] * n_per_group)
        y = y0 + true_ate * d
        
        # Test
        res = difference_in_means(y, d)
        if res['p_value'] < alpha:
            rejections += 1
    return rejections / n_sims

sample_sizes = np.arange(20, 520, 20)
powers = [simulate_power(n) for n in sample_sizes]

# Find the minimum n for 80% power
min_n_80 = None
for n, p in zip(sample_sizes, powers):
    if p >= 0.80:
        min_n_80 = n
        break

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sample_sizes, powers, 'o-', color='#3B4CCA', linewidth=2, markersize=5)
ax.axhline(0.80, color='#FFD733', linestyle='--', linewidth=2, label='80% power target')
if min_n_80 is not None:
    ax.axvline(min_n_80, color='#EE1515', linestyle=':', linewidth=2,
               label=f'Min N per group = {min_n_80}')
ax.set_xlabel('Sample Size per Group')
ax.set_ylabel('Empirical Power')
ax.set_title(f'Power Curve (True ATE = 0.3, alpha = 0.05, 1000 simulations each)')
ax.set_ylim(0, 1.05)
ax.legend(frameon=True, fontsize=11)
plt.tight_layout()
plt.show()

if min_n_80:
    print(f"Minimum sample size per group for 80% power: {min_n_80}")
    print(f"Total sample size needed: {2 * min_n_80}")

---
## Chapter Summary

In this chapter, you learned:

1. **Randomization** eliminates selection bias by making treatment independent of potential outcomes.
2. **Balance tables** verify that randomization produced comparable groups.
3. The **difference-in-means** estimator is unbiased for the ATE under randomization.
4. **Regression adjustment** (Lin, 2013) can improve precision without introducing bias.
5. **Randomization inference** provides exact p-values without distributional assumptions.
6. **Power analysis** determines the sample size needed to detect a given effect.
7. **Noncompliance** can be handled with ITT analysis or the Wald (LATE) estimator.
8. **Attrition** is dangerous when it is differential (related to treatment assignment).

You have now earned the **Boulder Badge** -- the foundation of causal inference design. Next, we travel to **Cerulean City** to learn about selection on observables and matching.

In [ ]:
badge_earned("Boulder", 2)